In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

# 1. Загрузка очищенных данных
df = pd.read_csv("../data/processed/dataset_clean.csv")

# 2. Создание новых признаков (Feature Engineering)
# Добавим осмысленный бизнес-признак: отношение дохода к возрасту
df["income_per_age"] = df["income"] / df["age"]

# 3. Выбор целевой переменной и матрицы признаков
target_column = "target"
X = df.drop(columns=[target_column, "id"]) # id удаляем, он не несет смысла для модели
y = df[target_column]

# 4. Разделение на обучающую (60%), валидационную (20%) и тестовую (20%) выборки
# Сначала бьем на train_full (80%) и test (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
# Теперь train_full бьем на train (60% от исходного) и val (20% от исходного)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

print(f"Размеры выборок: Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")

# 5. Создание конвейера предобработки
numeric_features = ["age", "income", "income_per_age"]
categorical_features = ["category"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Обучаем препроцессор ТОЛЬКО на X_train
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# 6. Сохранение выборок и конвейера
model_data_dir = Path("../data/model")
model_data_dir.mkdir(parents=True, exist_ok=True)
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

X_train.to_csv(model_data_dir / "X_train.csv", index=False, encoding="utf-8-sig")
X_val.to_csv(model_data_dir / "X_validation.csv", index=False, encoding="utf-8-sig")
X_test.to_csv(model_data_dir / "X_test.csv", index=False, encoding="utf-8-sig")
y_train.to_csv(model_data_dir / "y_train.csv", index=False)
y_val.to_csv(model_data_dir / "y_validation.csv", index=False)
y_test.to_csv(model_data_dir / "y_test.csv", index=False)

joblib.dump(preprocessor, models_dir / "feature_preprocessor.joblib")
print("Выборки и конвейер успешно сохранены.")

Размеры выборок: Train: (60, 4), Validation: (20, 4), Test: (20, 4)
Выборки и конвейер успешно сохранены.
